# Extending Elastic Agent Builder with LlamaExtract

We combine **LlamaExtract** (schema-driven PDF extraction) with **Elastic Agent Builder**
(workflow tools + search) to process complex documents like the
[World Bank Global Economic Prospects (Jan 2026)](https://www.worldbank.org/en/publication/global-economic-prospects).

Pipeline: Define schema > Create extraction agent > Upload PDF > Workflow (extract + index) > Create agent > Ask questions.

## Setup

In [ ]:
%pip install llama-cloud elasticsearch dotenv pydantic -q

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")

## Define the Extraction Schema

Pydantic model that defines what fields to extract from the PDF.

In [ ]:
from pydantic import BaseModel, Field


class EconomicReportSummary(BaseModel):
    report_title: str = Field(description="Full title of the report")
    publication_date: str = Field(
        description="Publication date in YYYY-MM format, e.g. '2026-01'"
    )
    frontier_market_gdp_share_pct: float = Field(
        description="Frontier markets' share of global GDP in 2025 as a percentage, "
        "extracted from Figure ES.A bar chart"
    )
    frontier_market_investment_growth_2020s_pct: float = Field(
        description="Average annual per capita investment growth for frontier markets "
        "in the early 2020s (2020-24), extracted from Figure ES.C bar chart, as a percentage"
    )
    executive_summary: str = Field(
        description="Concise summary of the report's main findings and conclusions"
    )
    key_vulnerabilities: str = Field(
        description="Main vulnerabilities and risks facing frontier markets, as a paragraph"
    )
    policy_recommendations: str = Field(
        description="Key policy recommendations for frontier market policymakers, as a paragraph"
    )


print(EconomicReportSummary.model_json_schema())

## Create the Extraction Agent

Create a LlamaExtract agent configured with our schema. Save the `agent.id` — you'll need it for the workflow.

In [ ]:
from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=LLAMA_CLOUD_API_KEY)

# Create the extraction agent with our schema
agent = client.extraction.extraction_agents.create(
    name="global-economic-extractor",
    data_schema=EconomicReportSummary.model_json_schema(),
    config={},
)
print(f"Created extraction agent: {agent.id}")

## Create the Elasticsearch Index

Create the index with mappings matching our extraction schema. The workflow will index documents here.

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected: {es.info()}")

In [ ]:
INDEX_NAME = "economic-reports"

mappings = {
    "mappings": {
        "properties": {
            "report_title": {"type": "keyword"},
            "publication_date": {"type": "date", "format": "yyyy-MM"},
            "frontier_market_gdp_share_pct": {"type": "float"},
            "frontier_market_investment_growth_2020s_pct": {"type": "float"},
            "executive_summary": {"type": "text"},
            "key_vulnerabilities": {"type": "text"},
            "policy_recommendations": {"type": "text"},
        }
    }
}

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mappings)
print(f"Created index: {INDEX_NAME}")

## Workflow Tool for Elastic Agent Builder

1. **upload_pdf:** uploads the PDF from a public URL to LlamaCloud.
2. **create_extraction_job:** starts the extraction job against the uploaded file.
3. **wait_for_extraction:** pauses 15s for LlamaExtract to process.
4. **get_extraction_result:** retrieves results, retrying every 10s up to 12 times.
5. **index_extracted_data:** indexes the extracted fields into Elasticsearch.
6. **verify_document:** confirms the document was indexed.

```yaml
name: LlamaExtract Economic Report Processor
description: >
  Uploads a PDF from a URL to LlamaCloud, runs extraction,
  and indexes the structured results into Elasticsearch.
enabled: true

inputs:
  - name: pdf_url
    type: string
    description: Public URL of the PDF to process
    required: true

consts:
  indexName: economic-reports
  llamaExtractBaseUrl: https://api.cloud.llamaindex.ai/api/v1
  extractionAgentId: 5bec3398-a0f2-4793-b175-761f29023816
  documentId: global-economic-prospects-jan-2026
  llamaCloudApiKey: llx-YOUR-API-KEY-HERE

triggers:
  - type: manual

steps:
  # Upload PDF from URL to LlamaCloud
  - name: upload_pdf
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/files/upload_from_url"
      method: PUT
      headers:
        Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
        Content-Type: application/json
      body: |
        {
          "url": "{{ inputs.pdf_url }}"
        }
      timeout: 60s

  - name: create_extraction_job
    type: http
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs"
      method: POST
      headers:
        Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
        Content-Type: application/json
      body: |
        {
          "extraction_agent_id": "{{ consts.extractionAgentId }}",
          "file_id": "{{ steps.upload_pdf.output.data.id }}"
        }
      timeout: 30s

  # Wait for the extraction job to process
  - name: wait_for_extraction
    type: wait
    with:
      duration: "15s"

  # Retrieve results — retries every 10s up to 12 times (2 min total)
  - name: get_extraction_result
    type: http
    on-failure:
      retry:
        max-attempts: 12
        delay: "10s"
    with:
      url: "{{ consts.llamaExtractBaseUrl }}/extraction/jobs/{{ steps.create_extraction_job.output.data.id }}/result"
      method: GET
      headers:
        Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
        Accept: application/json
      timeout: 120s

  - name: index_extracted_data
    type: elasticsearch.index
    with:
      index: "{{ consts.indexName }}"
      id: "{{ consts.documentId }}"
      document:
        report_title: "{{ steps.get_extraction_result.output.data.data.report_title }}"
        publication_date: "{{ steps.get_extraction_result.output.data.data.publication_date }}"
        frontier_market_gdp_share_pct: "{{ steps.get_extraction_result.output.data.data.frontier_market_gdp_share_pct }}"
        frontier_market_investment_growth_2020s_pct: "{{ steps.get_extraction_result.output.data.data.frontier_market_investment_growth_2020s_pct }}"
        executive_summary: "{{ steps.get_extraction_result.output.data.data.executive_summary }}"
        key_vulnerabilities: "{{ steps.get_extraction_result.output.data.data.key_vulnerabilities }}"
        policy_recommendations: "{{ steps.get_extraction_result.output.data.data.policy_recommendations }}"
      refresh: wait_for

  - name: verify_document
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        term:
          _id: "{{ consts.documentId }}"
```

## Create the Agent via API

Create a workflow tool that calls the extraction workflow, and two ES|QL tools to query the indexed results.
Then create an agent that uses all three tools. We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api).

**Note:** After creating the workflow, copy its ID from the Elastic UI and set it below as `WORKFLOW_ID`.

In [ ]:
import requests

WORKFLOW_ID = "workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0"  # Copy from Elastic UI after creating the workflow

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

print(WORKFLOW_ID)

In [ ]:
# Create the workflow tool
workflow_tool_payload = {
    "id": "run_llamaextract_workflow",
    "type": "workflow",
    "description": (
        "Triggers the LlamaExtract extraction workflow. "
        "Use this tool to extract structured data from a PDF URL and index it into Elasticsearch. "
        "Requires only the public URL of the PDF to process."
    ),
    "tags": ["llama-extract", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(response.json())

In [ ]:
# Create the structured indicators query tool
structured_query_payload = {
    "id": "query_structured_indicators",
    "type": "esql",
    "description": (
        "Query structured economic indicators using exact filters on numeric fields. "
        "Use this for questions like 'Which reports show frontier market GDP share below 5%?' "
        "or 'Show investment growth for reports published after 2025-01'."
    ),
    "tags": ["economic-data", "llama-extract"],
    "configuration": {
        "query": (
            "FROM economic-reports "
            "| WHERE frontier_market_gdp_share_pct <= ?max_gdp_share "
            "| KEEP report_title, publication_date, "
            "frontier_market_gdp_share_pct, frontier_market_investment_growth_2020s_pct "
            "| SORT publication_date DESC "
            "| LIMIT 10"
        ),
        "params": {
            "max_gdp_share": {
                "type": "double",
                "description": "Maximum frontier market GDP share percentage to filter by",
            }
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=structured_query_payload,
)
print(f"Structured query tool: {response.status_code}")
print(response.json())

# Create the narrative text search tool
text_search_payload = {
    "id": "search_economic_narratives",
    "type": "esql",
    "description": (
        "Search narrative content in economic reports. "
        "Use this for open-ended questions like 'What are the main risks for frontier markets?' "
        "or 'What does the report recommend for policymakers?'."
    ),
    "tags": ["economic-data", "llama-extract"],
    "configuration": {
        "query": (
            "FROM economic-reports "
            "| WHERE MATCH(executive_summary, ?query) "
            "OR MATCH(key_vulnerabilities, ?query) "
            "OR MATCH(policy_recommendations, ?query) "
            "| KEEP report_title, executive_summary, key_vulnerabilities, policy_recommendations "
            "| LIMIT 5"
        ),
        "params": {
            "query": {
                "type": "keyword",
                "description": "The search query to find relevant narrative content",
            }
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=text_search_payload,
)
print(f"Text search tool: {response.status_code}")
print(response.json())

In [ ]:
# Create the agent with all three tools
agent_payload = {
    "id": "economic-report-analyst",
    "name": "Economic Report Analyst",
    "description": "Extracts and analyzes economic reports from PDFs using LlamaExtract and Elasticsearch.",
    "labels": ["economics", "llama-extract"],
    "configuration": {
        "instructions": (
            "You are an economic research assistant. You have three tools:\n"
            "1. run_llamaextract_workflow: Use this FIRST to extract and index data from a PDF.\n"
            "2. query_structured_indicators: Use this for structured questions that filter on "
            "numeric fields like GDP share or investment growth.\n"
            "3. search_economic_narratives: Use this for open-ended questions about risks, "
            "vulnerabilities, or policy recommendations.\n"
            "When presenting data, use clear formatting with bullet points or tables. "
            "Always cite the report title and publication date."
        ),
        "tools": [
            {
                "tool_ids": [
                    "run_llamaextract_workflow",
                    "query_structured_indicators",
                    "search_economic_narratives",
                ]
            }
        ],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

In [ ]:
PDF_URL = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/supporting-blog-content/elastic-agent-builder-llamaindex-document-processing/Markets-Executive-Summary.pdf"  # Replace with actual PDF URL

# Ask the agent to extract and analyze the report
converse_payload = {
    "agent_id": "economic-report-analyst",
    "input": (
        f"Process this PDF and extract its data: {PDF_URL}\n\n"
        "Once extracted, answer: What are the main vulnerabilities facing frontier markets "
        "and what policy recommendations does the report suggest to address them?"
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=converse_payload,
)
print(json.dumps(response.json(), indent=2))

## Cleanup

In [ ]:
# Delete the Elasticsearch index
es.indices.delete(index=INDEX_NAME)

In [ ]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/economic-report-analyst", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/run_llamaextract_workflow", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/query_structured_indicators", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/search_economic_narratives", headers=headers
)